# Loan Performance Analysis
In this notebook, the goal is to conduct a loan performance analysis from the perspective of the lender, Lending Club. Our goal with this analysis is to extract insights related to assessed risk (via grades and subgrades), actual risk, and how various factors like term, income level, and purpose of loan are related to default risk.

Note: We have used Spark for the data analysis to ensure scaleability. Due to limited plotting capabilities with Spark, only aggregated, small dataframes have been converted to Pandas dataframes to be plotted using Seaborn. This does not harm scalability in a significant way because the aggregated dataframes are much smaller than the dataset itself.

#### Import Dependencies and Read Dataset

In [238]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, count, sum, lit
import pyspark.sql.functions as F
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import math


spark = SparkSession.builder \
    .appName("LoanPerformanceAnalysis") \
    .getOrCreate()

INPUT_PATH = "../dataset/processed"

df = spark.read.parquet(INPUT_PATH)

# Caching this as its our main dataframe that we will be using for analysis
df.cache()

print(f"Total rows: {df.count():,}")
df.show(5)

25/11/22 15:36:02 WARN CacheManager: Asked to cache already cached data.


Total rows: 160,000
+----+---------+---------+-----------+---------------+----+--------+-----------+-----+---------+--------------+----------+--------------+----------+-------------------+----------+-----------+----------+----+----+------------------+------------------+--------+----------+-----+-----------+----------------+--------------+----------------------+----------------------+--------+-------+---------+----------+---------+-------------------+---------+-------------+-----------+---------------+---------------+-------------+------------------+----------+-----------------------+------------+---------------+------------+------------------+--------------------------+---------------------------+-----------+----------------+----------------+---------+-------------------------+--------------+------------+-----------+-----------+-----------+-----------+-----------+------------------+------------+-------+-----------+-----------+----------+--------+----------------+------+-----------+----

#### Loan Performance Specific EDA
First, we want to perform some Exploratory Data Analysis (EDA) on columns in the dataset that affect loan performance including loan amount and loan status.

In [239]:
loan_amount_summary = df.select("loan_amnt").describe()
print("Loan Amount Summary:")
loan_amount_summary.show()

Loan Amount Summary:
+-------+------------------+
|summary|         loan_amnt|
+-------+------------------+
|  count|            160000|
|   mean|      16036.625625|
| stddev|10141.969255575217|
|    min|              1000|
|    max|             40000|
+-------+------------------+



In [240]:
#show distinct values for loan_status
loan_status_values = df.select("loan_status").distinct().collect()
print("Distinct Loan Status Values:")
for row in loan_status_values:
    print(row['loan_status'])

Distinct Loan Status Values:
Fully Paid
In Grace Period
Charged Off
Late (31-120 days)
Current
Late (16-30 days)


#### Creating Helper Columns 
To make our analysis easier, we are going to create some helper columns like default flag and loss amount.

For the default flag, we are making some assumptions to classify a loan. Any loan with the following statuses will be classified as a Default loan by our flag:
1) Default
2) Charged Off
3) Late (31-120 days)

Once we have created the column, we can also look at the overall default rate.

In [241]:
# For default flag
df = df.withColumn(
    "default_flag",
    when(col("loan_status").isin("Charged Off", "Default", "Late (31-120 days)"), 1).otherwise(0)
)

# Let's also check the overall default rate
df.agg(
    count("*").alias("total_loans"),
    sum("default_flag").alias("total_defaults"),
).withColumn(
    "default_rate",
    when(col("total_loans") != 0, col("total_defaults") / col("total_loans")).otherwise(None)
).show()

+-----------+--------------+------------+
|total_loans|total_defaults|default_rate|
+-----------+--------------+------------+
|     160000|           891|  0.00556875|
+-----------+--------------+------------+



In [242]:
# Let's create a column that shows the loss amount that the lender incurred for each loan 
df = df.withColumn("recoveries", F.col("recoveries").cast("double"))

# Loss Amount = Outstanding Principal - Recoveries (if defaulted)
df = df.withColumn(
    "loss_amt",
    when(col("default_flag") == 1, col("out_prncp") - col("recoveries"))
     .otherwise(lit(0))
)

#### Default Rate according to Grade and Subgrade
In this analysis, our goal is to identify how Lending Club’s credit grades map to realized default. Grades are meant to represent how risky a loan is. Let's find out whether lower grades actually have higher default rates or if they are a poor representation.

We will be calculating various results by doing aggregations. Since dataframe outputs are difficult to read, we will also add some convenient visualizations to showcase the insights.


In [243]:
default_by_grade = (
    df.groupBy("grade", "sub_grade")
      .agg(
          F.round(F.avg("int_rate"), 4).alias("avg_int_rate"),     
          F.round(F.avg("default_flag"), 4).alias("default_rate"), 
          F.round(F.avg("loss_amt"), 2).alias("avg_loss"),
          F.count("*").alias("num_loans")
      )
      .orderBy("grade", "sub_grade")
)

# Caching this dataframe as we will be using it multiple times to show results and later for plotting
default_by_grade.cache()

default_by_grade.show(200)


+-----+---------+------------+------------+--------+---------+
|grade|sub_grade|avg_int_rate|default_rate|avg_loss|num_loans|
+-----+---------+------------+------------+--------+---------+
|    A|       A1|      6.2337|      0.0011|   15.16|    11398|
|    A|       A2|      6.8071|      0.0011|   19.56|     7207|
|    A|       A3|      7.3426|      0.0025|   37.98|     7662|
|    A|       A4|      8.0032|      0.0018|   25.18|    11904|
|    A|       A5|      8.6107|      0.0017|   20.55|     9739|
|    B|       B1|     10.1886|      0.0025|   35.35|     8053|
|    B|       B2|     10.5787|      0.0037|   49.17|     9205|
|    B|       B3|     11.1646|      0.0054|   73.89|     7192|
|    B|       B4|     11.6622|      0.0033|   48.11|    11051|
|    B|       B5|     12.8411|       0.004|   73.34|     8721|
|    C|       C1|     13.5582|      0.0038|   58.87|     8619|
|    C|       C2|     14.4689|      0.0058|   66.94|     7596|
|    C|       C3|     15.0179|       0.007|  110.09|   

25/11/22 15:36:03 WARN CacheManager: Asked to cache already cached data.


##### Visualizing the Results

In [244]:
pdf = default_by_grade.toPandas()

plt.figure(figsize=(8, 4))
sns.barplot(
    data=pdf,
    x="sub_grade",
    y="avg_int_rate",
    hue="grade",
    palette="mako"
)

plt.title("Average Interest Rate by Sub-Grade")
plt.ylabel("Interest Rate (%)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [245]:
plt.figure(figsize=(8, 4))
sns.barplot(
    data=pdf,
    x="sub_grade",
    y="default_rate",
    palette="rocket",
    hue="grade",
)

plt.title("Default Rate by Grade")
plt.ylabel("Default Rate")
plt.xlabel("Loan Grade")
plt.tight_layout()
plt.show()


In [246]:
plt.figure(figsize=(8, 4))
sns.scatterplot(data=pdf, x="avg_int_rate", y="default_rate", hue="grade", s=80)
plt.title("Interest Rate vs Default Rate")
plt.xlabel("Average Interest Rate (%)")
plt.ylabel("Default Rate")
plt.tight_layout()
plt.show()


In [247]:
plt.figure(figsize=(8, 4))
sns.barplot(
    data=pdf,
    x="sub_grade",
    y="avg_loss",
    palette="mako",
    hue="grade",
)
plt.title("Average Loss Amount by Sub-Grade")
plt.xlabel("Sub-Grade")
plt.ylabel("Average Loss ($)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [248]:
plt.figure(figsize=(8, 4))
sns.barplot(
    data=pdf,
    x="sub_grade",
    y="num_loans",
    palette="viridis",
    hue="grade",
)
plt.title("Number of Loans by Sub-Grade")
plt.xlabel("Sub-Grade")
plt.ylabel("Loan Count")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


We should check if the trends we see our statistically significant. I'll use a z-test for this!

In [249]:
from statsmodels.stats.proportion import proportions_ztest

# Aggregate by grade in Spark. I'll only convert to Pandas when there's guaranteed to be a small dataframe
default_by_grade_level = (
    df.groupBy("grade")
      .agg(
          F.sum("default_flag").alias("num_defaults"),
          F.count("*").alias("num_loans")
      )
      .orderBy("grade")
)

pdf = default_by_grade_level.toPandas()

results = []
grades = pdf['grade'].tolist()

for i in range(len(grades)-1):
    count = [pdf.loc[i, 'num_defaults'], pdf.loc[i+1, 'num_defaults']]
    nobs  = [pdf.loc[i, 'num_loans'], pdf.loc[i+1, 'num_loans']]
    
    z_stat, p_val = proportions_ztest(count, nobs)
    results.append([grades[i], grades[i+1], z_stat, p_val])

pd.DataFrame(results, columns=["grade1", "grade2", "z_stat", "p_value"])


,grade1,grade2,z_stat,p_value
0,A,B,-6.143926,8.050640e-10
1,B,C,-5.607485,2.052873e-08
2,C,D,-5.205239,1.937475e-07
3,D,E,-5.333560,9.630547e-08
4,E,F,-3.795050,1.476135e-04
5,F,G,-2.897966,3.755908e-03


##### Summary and Interpretation
- Credit grades behave as expected: interest rates and default rates exhibit a clear, nearly monotonic increase from A → G.
- Subgrades provide meaningful discrimination: even within the same grade, higher-risk subgrades show higher PD and higher pricing.
- Loss severity rises sharply in lower grades:
  - A-grade LGD ≈ \$15–40  
  - D-grade LGD ≈ \$120–190  
  - G-grade LGD ≈ \$600–3500
- Lower-grade borrowers default more and also default harder as both PD and LGD increase together.
- The default rate vs. interest rate plot shows strong evidence of risk-based pricing efficiency.
- The loan count distribution shows Lending Club concentrates its portfolio heavily in grades A-C, minimizing exposure to highly risky segments.
- The p_values from the z-test are far below the typical 0.05 significant threshold, indicating that the increase in default rates as we move from higher to lower grades is statistically significant.

#### Risk-Adjusted Return Analysis (Pricing Efficiency)
Now, we compare interest rate charged vs realized losses to measure if certain borrower segments were systematically underpriced or overpriced.

The formula is: Risk-adjusted return = int_rate – default_rate × LGD

First, we will calculate the Loss Given Default (LGD) which is the percentage of the loan that the lender is expected to lose in the event that the borrower defaults, then we will use this to find the Risk-adjusted return. We can aggregate this over different grades and subgrades like before.

In [250]:
# Compute LGD = loss_amt / loan_amnt for defaulted loans
df_lgd = df.withColumn(
    "lgd",
    F.when(F.col("default_flag") == 1,
           F.col("loss_amt") / F.col("loan_amnt"))
     .otherwise(0)
)

risk_return = (
    df_lgd.groupBy("grade", "sub_grade")
          .agg(
            F.round(F.avg("int_rate"), 4).alias("avg_int_rate"),
            F.round(F.avg("default_flag"), 4).alias("default_rate"),
            F.round(F.avg("lgd"), 4).alias("avg_lgd"),
            F.round(F.avg(F.col("int_rate") - F.col("default_flag")*F.col("lgd")), 2).alias("risk_adj_return")
          )
          .orderBy("risk_adj_return")
)
risk_return.cache()

risk_return.show(200)


+-----+---------+------------+------------+-------+---------------+
|grade|sub_grade|avg_int_rate|default_rate|avg_lgd|risk_adj_return|
+-----+---------+------------+------------+-------+---------------+
|    A|       A1|      6.2337|      0.0011| 8.0E-4|           6.23|
|    A|       A2|      6.8071|      0.0011| 8.0E-4|           6.81|
|    A|       A3|      7.3426|      0.0025| 0.0024|           7.34|
|    A|       A4|      8.0032|      0.0018| 0.0015|            8.0|
|    A|       A5|      8.6107|      0.0017| 0.0013|           8.61|
|    B|       B1|     10.1886|      0.0025| 0.0018|          10.19|
|    B|       B2|     10.5787|      0.0037| 0.0031|          10.58|
|    B|       B3|     11.1646|      0.0054| 0.0043|          11.16|
|    B|       B4|     11.6622|      0.0033|  0.003|          11.66|
|    B|       B5|     12.8411|       0.004| 0.0037|          12.84|
|    C|       C1|     13.5582|      0.0038| 0.0034|          13.55|
|    C|       C2|     14.4689|      0.0058| 0.00

25/11/22 15:36:06 WARN CacheManager: Asked to cache already cached data.


In [251]:
pdf_risk = risk_return.toPandas()

plt.figure(figsize=(8, 4))
sns.barplot(
    data=pdf_risk,
    x="sub_grade",
    y="risk_adj_return",
    hue="grade",
    palette="coolwarm",
)
plt.title("Risk-Adjusted Return by Subgrade")
plt.ylabel("Risk-Adjusted Return (%)")
plt.xlabel("Subgrade")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


##### Summary and Interpretation
- Risk-adjusted returns increase from A → G, meaning high-risk borrowers are charged more than necessary relative to their expected losses.
- Default rate × LGD is very small relative to interest income for all subgrades, so expected losses are minor compared to interest earned.
- The lender earns higher net returns on riskier borrowers, indicating conservative pricing.
- There is absolutely no evidence of underpricing. Even the riskiest segments are profitable after accounting for defaults.
- The pricing model is internally consistent in overcompensating for risk, but is not risk-neutral. Because the net returns are higher for riskier loans, it’s overcompensating, not aiming for equal expected profit across all grades. 

Perhaps this explains why so few of the loans are from riskier grades, because Lending Club's pricing model charges them a premium beyond what is required to compensate for the default risk.

#### Purpose vs Default Rate
Now, we want to see: are loans borrowed for certain purposes usually defaulted on more than loans borrowed for other purposes?

In [252]:
purpose_perf = (
    df.groupBy("purpose")
      .agg(
          F.count("*").alias("num_loans"),
          F.round(F.avg("int_rate"), 4).alias("avg_rate"),
          F.round(F.avg("default_flag"), 4).alias("default_rate"),
          F.round(F.avg("loss_amt"), 4).alias("avg_loss")
      )
      .orderBy(F.desc("default_rate"))
)
purpose_perf.cache()

purpose_perf.show(100)


+------------------+---------+--------+------------+--------+
|           purpose|num_loans|avg_rate|default_rate|avg_loss|
+------------------+---------+--------+------------+--------+
|            moving|      830|  14.957|      0.0181|179.0595|
|          vacation|      969| 14.6779|      0.0134|130.5579|
|    major_purchase|     2860| 14.1483|      0.0122|202.6846|
|  renewable_energy|       84|  13.383|      0.0119|202.8756|
|             other|     8620| 15.3631|      0.0117|141.6497|
|           medical|     1817| 14.4973|      0.0116|116.7342|
|    small_business|     1280| 14.6884|      0.0094|136.1122|
|             house|      995| 15.2512|       0.008|114.7356|
|  home_improvement|     9375| 13.2951|      0.0071|104.6046|
|debt_consolidation|    88181| 13.2125|      0.0052| 76.7161|
|               car|     1283| 13.6111|      0.0039| 65.1969|
|       credit_card|    43706| 11.3491|      0.0035| 52.7004|
+------------------+---------+--------+------------+--------+



25/11/22 15:36:07 WARN CacheManager: Asked to cache already cached data.


In [253]:
#Plotting Purpose Performance
pdf_purpose = purpose_perf.toPandas()
plt.figure(figsize=(8, 4))
sns.barplot(
    data=pdf_purpose,
    x="purpose",
    y="default_rate",
    palette="husl",
    hue="purpose",
)
plt.title("Default Rate by Loan Purpose")
plt.ylabel("Default Rate")
plt.xlabel("Loan Purpose")
plt.xticks(rotation=45)
plt.tight_layout()

In [254]:
#Plotting interest rate by purpose
plt.figure(figsize=(8, 4))
sns.barplot(
    data=pdf_purpose,
    x="purpose",
    y="avg_rate",
    palette="Set2",
    hue="purpose",
)
plt.title("Average Interest Rate by Loan Purpose")
plt.ylabel("Average Interest Rate (%)")
plt.xlabel("Loan Purpose")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

##### Summary and Interpretation
It seems that certain loan purposes correlate to higher default rates. However, it seems that the interest rate for the loan is decided without factoring in the purpose of the loan, because we do not see a correlation of the interest rate with the purpose of the loan.

#### Impact of Borrower DTI on Loan Default and Losses
Debt-to-Income ratio (DTI) is used as a common creditworthiness measure by lenders. A higher DTI means a borrower has more debt relative to income, and we want to explore if this significantly affects default risk.

In [255]:
# DTI buckets
df_dti = df.withColumn(
    "dti_bucket",
    F.when(F.col("dti") < 10, "<10")
     .when((F.col("dti") >= 10) & (F.col("dti") < 20), "10-20")
     .when((F.col("dti") >= 20) & (F.col("dti") < 35), "20-35")
     .otherwise(">35")
)

dti_order = {"<10": 1, "10-20": 2, "20-35": 3, ">35": 4}

df_dti_ordered = df_dti.withColumn(
    "dti_order",
    F.create_map([F.lit(x) for t in dti_order.items() for x in t]).getItem(F.col("dti_bucket"))
)

# Group and order including dti_order
term_dti = (
    df_dti_ordered.groupBy("term", "dti_bucket", "dti_order")
          .agg(
              F.count("*").alias("num_loans"),
              F.round(F.avg("default_flag"), 4).alias("default_rate"),
              F.round(F.avg("loss_amt"), 2).alias("avg_loss")
          )
          .orderBy("term", "dti_order")
          .drop("dti_order")
)

# Caching because I have to show this df and then plot by converting to pandas
term_dti.cache()

term_dti.show(100)


+----+----------+---------+------------+--------+
|term|dti_bucket|num_loans|default_rate|avg_loss|
+----+----------+---------+------------+--------+
|  36|       <10|    22366|      0.0073|   89.59|
|  36|     10-20|    42709|      0.0044|   56.35|
|  36|     20-35|    37327|      0.0056|   57.17|
|  36|       >35|     7264|      0.0059|    76.2|
|  60|       <10|     7598|      0.0079|  180.64|
|  60|     10-20|    19658|       0.005|   91.93|
|  60|     20-35|    19057|      0.0055|  102.41|
|  60|       >35|     4021|       0.006|  123.64|
+----+----------+---------+------------+--------+



/Applications/anaconda3/lib/python3.9/site-packages/pyspark/sql/classic/column.py:359: FutureWarning: A column as 'key' in getItem is deprecated as of Spark 3.0, and will not be supported in the future release. Use `column[key]` or `column.key` syntax instead.
  warnings.warn(
25/11/22 15:36:08 WARN CacheManager: Asked to cache already cached data.


In [256]:
pdf_dti = term_dti.toPandas()

dti_order = ["<10", "10-20", "20-35", ">35"]
pdf_dti['dti_bucket'] = pd.Categorical(pdf_dti['dti_bucket'], categories=dti_order, ordered=True)

plt.figure(figsize=(8,4))
sns.barplot(
    data=pdf_dti,
    x="dti_bucket",
    y="default_rate",
    hue="term",
    palette="Paired"
)
plt.title("Default Rate by DTI Bucket and Loan Term")
plt.ylabel("Default Rate")
plt.xlabel("DTI Bucket")
plt.tight_layout()
plt.show()


In [257]:
plt.figure(figsize=(8,4))
sns.barplot(
    data=pdf_dti,
    x="dti_bucket",
    y="avg_loss",
    hue="term",
    palette="Paired"
)
plt.title("Average Loss by DTI Bucket and Loan Term")
plt.ylabel("Average Loss ($)")
plt.xlabel("DTI Bucket")
plt.tight_layout()
plt.show()


##### Summary and Interpretation
- Borrowers with **very low DTI (<10%)** surprisingly have higher default rates and losses than those in all other ranges. This is slightly unusual as the default rate and average loss increases from the 10-20% range onwards.
   - One potential explanation could be that these are very low-income borrowers who have limited financial flexibility and struggle to pay back even smaller loands.

- Borrowers with high DTI (>35%) have slightly elevated default rates and significantly higher losses, especially for long-term (60-month) loans.
- Longer-term loans consistently have higher losses across all DTI buckets, which can be linked to the higher exposure duration.
- Interaction between loan term and DTI is important for risk management: high DTI + long term → higher expected losses.


#### Installment to Income Ratio
Previously, we worked with the DTI which is the ratio of the total debt to the income. However, we can do a more granular analysis by using just the current loan's installment and comparing that to annual income to find a correlation with the default risk and loss for Lending Club.


In [258]:
# # Cast numeric columns
df = df.withColumn("installment", F.col("installment").cast("double"))
df = df.withColumn("annual_inc", F.col("annual_inc").cast("double"))

df = df.filter(F.col("annual_inc") > 0)

# Installment to Income Ratio (ITIR)
df_itir = df.withColumn(
    "itir",
    F.try_divide(F.col("installment"), F.col("annual_inc") / 12)
)

# Let's create ITIR buckets
df_itir = df_itir.withColumn(
    "itir_bucket",
    F.when(F.col("itir") < 0.05, "<5%")
     .when(F.col("itir") < 0.10, "5-10%")
     .when(F.col("itir") < 0.20, "10-20%")
     .otherwise(">20%")
)

#Making a temporary column to help with ordering
itir_order = {"<5%": 1, "5-10%": 2, "10-20%": 3, ">20%": 4}
df_itir = df_itir.withColumn(
    "itir_order",
    F.create_map([F.lit(x) for t in itir_order.items() for x in t]).getItem(F.col("itir_bucket"))
)

itir_summary = (
    df_itir.groupBy("itir_bucket", "term", "itir_order")
           .agg(
               F.round(F.avg("default_flag"), 4).alias("default_rate"),
               F.round(F.avg("loss_amt"), 2).alias("avg_loss"),
               F.round(F.avg("loan_amnt"), 2).alias("avg_loan_amt"),
               F.count("*").alias("num_loans")
           )
           .orderBy("term", "itir_order", "itir_bucket")
)

itir_summary.show(100)

/Applications/anaconda3/lib/python3.9/site-packages/pyspark/sql/classic/column.py:359: FutureWarning: A column as 'key' in getItem is deprecated as of Spark 3.0, and will not be supported in the future release. Use `column[key]` or `column.key` syntax instead.
  warnings.warn(


+-----------+----+----------+------------+--------+------------+---------+
|itir_bucket|term|itir_order|default_rate|avg_loss|avg_loan_amt|num_loans|
+-----------+----+----------+------------+--------+------------+---------+
|        <5%|  36|         1|      0.0054|   27.19|     7473.58|    38340|
|      5-10%|  36|         2|      0.0048|   59.43|     13746.7|    41579|
|     10-20%|  36|         3|      0.0063|  111.63|    19782.54|    24923|
|       >20%|  36|         4|      0.0088|  172.03|    23810.24|     4660|
|        <5%|  60|         1|       0.004|   58.86|    17398.98|    10877|
|      5-10%|  60|         2|      0.0054|   99.57|    21494.94|    23801|
|     10-20%|  60|         3|       0.007|  159.41|    25311.44|    13362|
|       >20%|  60|         4|      0.0102|   226.0|    27250.73|     2161|
+-----------+----+----------+------------+--------+------------+---------+



In [259]:
pdf = itir_summary.toPandas()

pdf["term"] = pdf["term"].astype(str)   #this is so that seaborn can treat it as categorical variable

itir_order = ["<5%", "5-10%", "10-20%", ">20%"]

plt.figure(figsize=(8,4))
sns.barplot(
    data=pdf,
    x="itir_bucket",
    y="default_rate",
    hue="term",
    order=itir_order,      
    palette="mako"
)

plt.title("Default Rate by ITIR Bucket and Term")
plt.ylabel("Default Rate")
plt.xlabel("Installment-to-Income Buckets (ITIR)")
plt.xticks(rotation=0)
plt.legend(title="Term (months)")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [260]:
plt.figure(figsize=(8,4))
sns.barplot(
    data=pdf,
    x="itir_bucket",
    y="avg_loss",
    hue="term",
    order=itir_order, 
    palette="mako"
)

plt.title("Average Loss by ITIR Bucket and Term")
plt.ylabel("Average Loss")
plt.xlabel("Installment-to-Income Buckets (ITIR)")
plt.xticks(rotation=0)
plt.legend(title="Term (months)")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

#### Summary and Interpretation
- Default risk generally increases as ITIR increases, especially for long-term loans. This can be interpreted such that longer loans amplify risk for high ITIR borrowers. Borrowers spending >20% of income on installments have the highest default rates.
- Average loss increases drastically as ITIR increases, and is higher for long term loans than it is for short-term loans. High ITIR borrowers not only default more often, but also have larger losses (they tend to borrow more relative to their income, and longer-term loans accumulate higher total principal).
In conclusion, ITIR is predictive of default risk. Moreover, for the same ITIR bucket, 60-month loans have higher losses than 36-month loans. Longer-term commitments exacerbate default severity. For optimal risk management, Lending Club can consider capping ITIR or adjusting interest rates to price this risk.